In [4]:
"""
02_bug_diagnostics.ipynb
========================
Two diagnostics, nothing more:

  1. Rank every feature in the Stage 4 FULL-MOMENTS datasets by its maximum
     |z| BEFORE the +/-5 clip, most-to-least, to see how bad the near-zero-std
     blow-up is. Only the full-moments files are scanned; the means files are a
     subset of their columns. Weekly features are NOT recomputed -- values are
     read directly from the saved Stage 4 parquet, so this measures the Stage 3
     expanding z-score as it actually landed on disk.

  2. Report the real first TRAINING date for each split in the aggregate splits,
     to confirm it lands ~2007-07-31 / 2007-08-31 (pushed there by the CFTC data
     starting 2006-07-03 plus the 252-day expanding-z warm-up).

Read-only. Writes nothing, changes no pipeline files.
"""

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("../..")

# Stage 4 saved model-ready datasets (already z-scored in Stage 3, already
# clipped? -- NO: Stage 4 is pre-clip; the +/-5 clip happens in 01_prepare_datasets.
# So these files still carry the true pre-clip z-scores, which is what we want.)
STAGE4_DIR = PROJECT_ROOT / "Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed"

FULL_MOMENTS_FILES = {
    "daily_full_moments":    "model_market_daily_full_moments.parquet",
    "combined_full_moments": "model_market_combined_full_moments.parquet",
}

# Columns that are never z-scored / not features -- excluded from the scan.
NON_FEATURE = {
    "date", "target_daily_return", "target_monthly_return",
    "minret_5d", "minret_5d_z", "y_binary",
}
BINARY_FEATURES = {
    "vix_above_20", "vix_above_30",
    "curve_inverted_2y10y", "curve_inverted_3m10y", "credit_stress",
}

# The prepared aggregate splits (to read real first training dates).
SPLITS_DIR = PROJECT_ROOT / "Data/Splits"
SPLIT_NAMES = ["Split_B", "Split_C", "Split_A", "Split_D"]


# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 1: rank full-moments features by pre-clip max |z|
# ═══════════════════════════════════════════════════════════════════════════════

def rank_extreme_zscores():
    print("=" * 78)
    print("DIAGNOSTIC 1: features ranked by pre-clip max |z|  (full-moments only)")
    print("=" * 78)

    for label, fname in FULL_MOMENTS_FILES.items():
        path = STAGE4_DIR / fname
        print(f"\n{'-' * 78}")
        print(f"{label}   ({fname})")
        print(f"{'-' * 78}")

        if not path.exists():
            print(f"  MISSING: {path}")
            continue

        df = pd.read_parquet(path)

        feat_cols = [
            c for c in df.columns
            if c not in NON_FEATURE and c not in BINARY_FEATURES
        ]
        # numeric only (skip any stray object/string columns)
        feat_cols = [c for c in feat_cols
                     if pd.api.types.is_numeric_dtype(df[c])]

        # cast nullable dtypes so abs()/comparisons behave
        sub = df[feat_cols].astype("float64")

        abs_max = sub.abs().max()
        n_gt5   = (sub.abs() > 5).sum()
        n_gt10  = (sub.abs() > 10).sum()
        n_gt50  = (sub.abs() > 50).sum()
        n_valid = sub.notna().sum()

        summary = pd.DataFrame({
            "max_abs_z": abs_max,
            "n_gt5":  n_gt5,
            "n_gt10": n_gt10,
            "n_gt50": n_gt50,
            "n_valid": n_valid,
        }).sort_values("max_abs_z", ascending=False)

        n_feat = len(feat_cols)
        print(f"  Features scanned: {n_feat}")
        print(f"  Any |z| > 5 : {int((abs_max > 5).sum())} features")
        print(f"  Any |z| > 10: {int((abs_max > 10).sum())} features")
        print(f"  Any |z| > 50: {int((abs_max > 50).sum())} features")
        print(f"  Global max |z|: {abs_max.max():.1f}  ({abs_max.idxmax()})")

        print(f"\n  Top 40 by max |z| (most -> least):")
        with pd.option_context("display.max_rows", None,
                               "display.width", 120):
            print(summary.head(40).to_string(
                formatters={"max_abs_z": "{:.1f}".format}))

        # Save the full ranking to CSV next to this notebook for inspection,
        # so you can scroll all of it rather than just the top 40.
        out = Path(f"zscore_ranking_{label}.csv")
        summary.to_csv(out)
        print(f"\n  Full ranking ({n_feat} rows) written to: {out.resolve()}")

        # For combined_full_moments only, two extra views: which features have
        # the MOST values past the wall. A feature high here (rather than in the
        # max|z| table) is broadly mis-scaled -- it dumps many observations onto
        # the +/-5 grid edge -- versus one with a single freak spike.
        if label == "combined_full_moments":
            print(f"\n  Top 40 by COUNT of |z| > 5 (most values past the wall):")
            by_gt5 = summary.sort_values("n_gt5", ascending=False)
            with pd.option_context("display.max_rows", None, "display.width", 120):
                print(by_gt5.head(40).to_string(
                    formatters={"max_abs_z": "{:.1f}".format}))

            print(f"\n  Top 40 by COUNT of |z| > 10:")
            by_gt10 = summary.sort_values("n_gt10", ascending=False)
            with pd.option_context("display.max_rows", None, "display.width", 120):
                print(by_gt10.head(40).to_string(
                    formatters={"max_abs_z": "{:.1f}".format}))


# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 2: real first training date per split
# ═══════════════════════════════════════════════════════════════════════════════

def report_first_training_dates():
    print("\n\n" + "=" * 78)
    print("DIAGNOSTIC 2: real first TRAINING date per split")
    print("=" * 78)
    print("  (expected ~2007-07-31 / 2007-08-31: CFTC starts 2006-07-03, then")
    print("   the 252-day expanding-z warm-up pushes the first valid row later)")

    # Use full_moments train files; either feature set has the same date spine.
    for split in SPLIT_NAMES:
        train_path = SPLITS_DIR / split / "full_moments_train.parquet"
        if not train_path.exists():
            print(f"\n  {split}: MISSING {train_path}")
            continue
        d = pd.read_parquet(train_path, columns=["date"])
        d["date"] = pd.to_datetime(d["date"])
        print(f"\n  {split}:")
        print(f"    train first date: {d['date'].min().date()}")
        print(f"    train last  date: {d['date'].max().date()}")
        print(f"    train rows:       {len(d):,}")


if __name__ == "__main__":
    rank_extreme_zscores()
    report_first_training_dates()

DIAGNOSTIC 1: features ranked by pre-clip max |z|  (full-moments only)

------------------------------------------------------------------------------
daily_full_moments   (model_market_daily_full_moments.parquet)
------------------------------------------------------------------------------
  Features scanned: 1132
  Any |z| > 5 : 960 features
  Any |z| > 10: 473 features
  Any |z| > 50: 52 features
  Global max |z|: 22648.4  (ivol_q_spread)

  Top 40 by max |z| (most -> least):
                                   max_abs_z  n_gt5  n_gt10  n_gt50  n_valid
ivol_q_spread                        22648.4      7       3       1     4299
dollarrealizedspread_lr_dw_cwmean     8666.0      4       2       1     4299
dollarrealizedspread_lr_dw_cwstd      8298.6      3       2       2     4299
effectivespread_dollar_dw_cwmean      6973.1      3       3       1     4299
effectivespread_dollar_dw_cwstd       5588.1      3       2       2     4299
venue_range_a_cwstd                   3796.9      5  

In [5]:
"""
02_bug_diagnostics.ipynb
========================
Two diagnostics, nothing more:

  1. Rank every feature in the Stage 4 FULL-MOMENTS datasets by its maximum
     |z| BEFORE the +/-5 clip, most-to-least, to see how bad the near-zero-std
     blow-up is. Only the full-moments files are scanned; the means files are a
     subset of their columns. Weekly features are NOT recomputed -- values are
     read directly from the saved Stage 4 parquet, so this measures the Stage 3
     expanding z-score as it actually landed on disk.

  2. Report the real first TRAINING date for each split in the aggregate splits,
     to confirm it lands ~2007-07-31 / 2007-08-31 (pushed there by the CFTC data
     starting 2006-07-03 plus the 252-day expanding-z warm-up).

Read-only. Writes nothing, changes no pipeline files.
"""

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("../..")

# Stage 4 saved model-ready datasets (already z-scored in Stage 3, already
# clipped? -- NO: Stage 4 is pre-clip; the +/-5 clip happens in 01_prepare_datasets.
# So these files still carry the true pre-clip z-scores, which is what we want.)
STAGE4_DIR = PROJECT_ROOT / "Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed"

FULL_MOMENTS_FILES = {
    "daily_full_moments":    "model_market_daily_full_moments.parquet",
    "combined_full_moments": "model_market_combined_full_moments.parquet",
}

# Columns that are never z-scored / not features -- excluded from the scan.
NON_FEATURE = {
    "date", "target_daily_return", "target_monthly_return",
    "minret_5d", "minret_5d_z", "y_binary",
}
BINARY_FEATURES = {
    "vix_above_20", "vix_above_30",
    "curve_inverted_2y10y", "curve_inverted_3m10y", "credit_stress",
}

# The prepared aggregate splits (to read real first training dates).
SPLITS_DIR = PROJECT_ROOT / "Data/Splits"
SPLIT_NAMES = ["Split_B", "Split_C", "Split_A", "Split_D"]


# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 1: rank full-moments features by pre-clip max |z|
# ═══════════════════════════════════════════════════════════════════════════════

def rank_extreme_zscores():
    print("=" * 78)
    print("DIAGNOSTIC 1: features ranked by pre-clip max |z|  (full-moments only)")
    print("=" * 78)

    for label, fname in FULL_MOMENTS_FILES.items():
        path = STAGE4_DIR / fname
        print(f"\n{'-' * 78}")
        print(f"{label}   ({fname})")
        print(f"{'-' * 78}")

        if not path.exists():
            print(f"  MISSING: {path}")
            continue

        df = pd.read_parquet(path)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)

        feat_cols = [
            c for c in df.columns
            if c not in NON_FEATURE and c not in BINARY_FEATURES
        ]
        # numeric only (skip any stray object/string columns)
        feat_cols = [c for c in feat_cols
                     if pd.api.types.is_numeric_dtype(df[c])]

        # cast nullable dtypes so abs()/comparisons behave
        sub = df[feat_cols].astype("float64")

        # ── Per feature: dedup CONSECUTIVE identical values before counting ──
        # A monthly feature forward-filled to daily repeats each z-score ~21x,
        # which inflates "count past the wall" ~21-fold vs a genuinely daily
        # feature. Collapsing runs of identical consecutive values counts each
        # monthly print once. (Consecutive, not global unique() -- two distinct
        # months could coincidentally share a value and must not be merged.)
        #
        # Also classify each feature as monthly-like: if its longest run of
        # identical consecutive values exceeds 14 days it can't be daily. This
        # is a heuristic; the `monthly_` name prefix is the ground truth, so
        # both are reported for eyeballing.
        rows = {}
        for c in feat_cols:
            v = sub[c].values
            valid = ~np.isnan(v)
            vv = v[valid]
            if len(vv) == 0:
                rows[c] = dict(max_abs_z=np.nan, n_gt5=0, n_gt10=0, n_gt50=0,
                               n_unique_obs=0, max_run=0, is_monthly=False,
                               n_daily_obs=0)
                continue

            # run boundaries: first element, then every change from previous
            change = np.empty(len(vv), dtype=bool)
            change[0] = True
            change[1:] = vv[1:] != vv[:-1]
            dedup = vv[change]                     # one value per consecutive run

            # longest run length (for monthly classification)
            run_ids = np.cumsum(change)
            _, run_lengths = np.unique(run_ids, return_counts=True)
            max_run = int(run_lengths.max())
            is_monthly = max_run > 14

            a = np.abs(dedup)
            rows[c] = dict(
                max_abs_z=float(a.max()),
                n_gt5=int((a > 5).sum()),
                n_gt10=int((a > 10).sum()),
                n_gt50=int((a > 50).sum()),
                n_unique_obs=int(len(dedup)),      # deduped observation count
                max_run=max_run,
                is_monthly=is_monthly,
                n_daily_obs=int(len(vv)),          # raw (pre-dedup) count
            )

        summary = pd.DataFrame(rows).T
        for col in ["max_abs_z", "n_gt5", "n_gt10", "n_gt50",
                    "n_unique_obs", "max_run", "n_daily_obs"]:
            summary[col] = pd.to_numeric(summary[col])
        summary = summary.sort_values("max_abs_z", ascending=False)

        n_feat = len(feat_cols)
        n_monthly = int(summary["is_monthly"].sum())
        print(f"  Features scanned: {n_feat}  "
              f"(classified monthly-like by max run >14d: {n_monthly}, "
              f"daily-like: {n_feat - n_monthly})")
        print(f"  Counts below are DEDUPED (each consecutive run counted once),"
              f" so a monthly feature's 21x repeats count as 1.")
        print(f"  Any |z| > 5 : {int((summary['n_gt5']  > 0).sum())} features")
        print(f"  Any |z| > 10: {int((summary['n_gt10'] > 0).sum())} features")
        print(f"  Any |z| > 50: {int((summary['n_gt50'] > 0).sum())} features")
        print(f"  Global max |z|: {summary['max_abs_z'].max():.1f}  "
              f"({summary['max_abs_z'].idxmax()})")

        show = ["max_abs_z", "n_gt5", "n_gt10", "n_gt50",
                "n_unique_obs", "max_run", "is_monthly"]

        print(f"\n  Top 40 by max |z| (most -> least):")
        with pd.option_context("display.max_rows", None,
                               "display.width", 140):
            print(summary[show].head(40).to_string(
                formatters={"max_abs_z": "{:.1f}".format}))

        # Save the full ranking to CSV next to this notebook for inspection,
        # so you can scroll all of it rather than just the top 40.
        out = Path(f"zscore_ranking_{label}.csv")
        summary.to_csv(out)
        print(f"\n  Full ranking ({n_feat} rows) written to: {out.resolve()}")

        # For combined_full_moments only, extra views: which features have the
        # MOST values past the wall (DEDUPED). A feature high here (rather than
        # in the max|z| table) is broadly mis-scaled -- it dumps many distinct
        # observations onto the +/-5 grid edge -- versus one freak spike. And a
        # monthly-only view, since your hunch is that a monthly feature's high
        # raw count may really be just one or two corrupt months.
        if label == "combined_full_moments":
            print(f"\n  Top 40 by COUNT of |z| > 5  (DEDUPED, most past the wall):")
            by_gt5 = summary.sort_values("n_gt5", ascending=False)
            with pd.option_context("display.max_rows", None, "display.width", 140):
                print(by_gt5[show].head(40).to_string(
                    formatters={"max_abs_z": "{:.1f}".format}))

            print(f"\n  Top 40 by COUNT of |z| > 10  (DEDUPED):")
            by_gt10 = summary.sort_values("n_gt10", ascending=False)
            with pd.option_context("display.max_rows", None, "display.width", 140):
                print(by_gt10[show].head(40).to_string(
                    formatters={"max_abs_z": "{:.1f}".format}))

            print(f"\n  Top 40 MONTHLY-ONLY by COUNT of |z| > 5  (DEDUPED):")
            monthly = summary[summary["is_monthly"]].sort_values(
                "n_gt5", ascending=False)
            print(f"    ({len(monthly)} monthly-like features; if n_gt5 is tiny "
                  f"here, a monthly feature's scary raw count was just 1-2 bad months)")
            with pd.option_context("display.max_rows", None, "display.width", 140):
                print(monthly[show].head(40).to_string(
                    formatters={"max_abs_z": "{:.1f}".format}))


# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 2: real first training date per split
# ═══════════════════════════════════════════════════════════════════════════════

def report_first_training_dates():
    print("\n\n" + "=" * 78)
    print("DIAGNOSTIC 2: real first TRAINING date per split")
    print("=" * 78)
    print("  (expected ~2007-07-31 / 2007-08-31: CFTC starts 2006-07-03, then")
    print("   the 252-day expanding-z warm-up pushes the first valid row later)")

    # Use full_moments train files; either feature set has the same date spine.
    for split in SPLIT_NAMES:
        train_path = SPLITS_DIR / split / "full_moments_train.parquet"
        if not train_path.exists():
            print(f"\n  {split}: MISSING {train_path}")
            continue
        d = pd.read_parquet(train_path, columns=["date"])
        d["date"] = pd.to_datetime(d["date"])
        print(f"\n  {split}:")
        print(f"    train first date: {d['date'].min().date()}")
        print(f"    train last  date: {d['date'].max().date()}")
        print(f"    train rows:       {len(d):,}")


if __name__ == "__main__":
    rank_extreme_zscores()
    report_first_training_dates()

DIAGNOSTIC 1: features ranked by pre-clip max |z|  (full-moments only)

------------------------------------------------------------------------------
daily_full_moments   (model_market_daily_full_moments.parquet)
------------------------------------------------------------------------------
  Features scanned: 1132  (classified monthly-like by max run >14d: 0, daily-like: 1132)
  Counts below are DEDUPED (each consecutive run counted once), so a monthly feature's 21x repeats count as 1.
  Any |z| > 5 : 960 features
  Any |z| > 10: 473 features
  Any |z| > 50: 52 features
  Global max |z|: 22648.4  (ivol_q_spread)

  Top 40 by max |z| (most -> least):
                                   max_abs_z  n_gt5  n_gt10  n_gt50  n_unique_obs  max_run is_monthly
ivol_q_spread                        22648.4      7       3       1          4299        1      False
dollarrealizedspread_lr_dw_cwmean     8666.0      4       2       1          4299        1      False
dollarrealizedspread_lr_dw_cwstd  